# 第 3 课 · 校验器、训练集与独立评估

**本课用时约 35 分钟。** 理解反馈循环里的答案校验和数据分区，并亲自检查数据泄漏。

## 学完后你能做到

- 用规则计算答案是否正确。
- 解释训练、开发、测试数据分别做什么。
- 理解训练奖励正确时为何还要固定测试题。

## 跟着操作

运行内置自检，试着把错误答案改为正确答案，再观察校验结果。

运行代码格时，把光标放在格子里，按 **Shift + Enter**。看到输出后再运行下一格。若第一次误点或修改了上方变量，选 Jupyter 菜单里的 **Kernel → Restart Kernel and Run All Cells**，从第一格重新运行。本课不要求预先懂强化学习，也不需要 GPU。

## 校验器决定什么能教会策略

本练习有精确的规则校验器。它将模型预测与指令对应的正确数字比较。通过的样本给所用规则增加权重，未通过的样本不更新。这代表一种带外部反馈的学习循环。

真实语言任务的奖励通常更难设计。若校验器只看 JSON 格式，语法正确但事实错误的答案也可能获奖励。后面的练习会专门检查这个反馈漏洞。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from rsi_lab import TRAIN, DEV, TEST, answer, verified

examples = [(8, 2, 1), (8, 2, 9), (0, 3, 9)]
for x, task, prediction in examples:
    print(x, task, prediction, "通过" if verified(x, task, prediction) else "未通过")
assert [verified(*item) for item in examples] == [True, False, True]

把 100 个起点划成三个互斥部分。训练集可用于产生训练样本，开发集用于观察每轮变化，测试集留到策略选择完成之后作最终报告。这次按起点连续分段。这些数字题存在周期结构，因此它只能演示隔离流程，不能证明真实任务的分布泛化。

In [ ]:
print("训练:", min(TRAIN), max(TRAIN), len(TRAIN))
print("开发:", min(DEV), max(DEV), len(DEV))
print("测试:", min(TEST), max(TEST), len(TEST))
assert not set(TRAIN) & set(DEV)
assert not set(TRAIN) & set(TEST)
assert not set(DEV) & set(TEST)
assert len(TRAIN) + len(DEV) + len(TEST) == 100

## 做一个校验器反例

给验证函数一个错误答案。如果它仍通过，就意味着奖励信号并未识别这项错误，后续优化可能朝错误目标前进。这里使用精确算术校验，预期错误答案必须被拒绝。

In [ ]:
x, task, wrong = 8, 2, 9
assert wrong != answer(x, task)
assert not verified(x, task, wrong)
print("错误答案被拒绝。训练反馈与任务正确性在这个小环境中一致。")

## 自检

- [ ] 三个集合没有重叠。
- [ ] 错误答案被校验器拒绝。
- [ ] 我能解释为什么每轮比较要用同一套开发题。
- [ ] 我知道最终测试集不应用来反复挑选参数。

下一课会逐轮运行训练，并从记录里读出反馈过程。

[返回课程首页并选择下一节](recursive_self_improvement.ipynb)